# VICOS BJJ: re-extract keypoints with YOLO11-pose (Kaggle GPU)

**Why:** the VICOS keypoints came from the authors' own detector. Phases 4–6 will run *our* detector on UFC footage,
so the classifier must be trained on keypoints from that same detector. This notebook re-runs YOLO11-pose (and optionally RTMO)
on every VICOS image and writes a JSON in the exact VICOS format, so `src/train_baseline.py --annotations ...` works unchanged.

**Settings (right sidebar):** Accelerator = **GPU T4 x1**, Internet = **On** (needs a phone-verified Kaggle account).

**Runtime estimate:** ~25–40 min download + unzip, ~20–40 min inference at `STRIDE=1` with `yolo11m-pose` (an estimate; it uses roughly 1 of your ~30 weekly GPU hours).
Set `STRIDE=3` for a ~3x faster first pass.

**Licenses:** VICOS data CC BY-NC-SA 4.0. Ultralytics YOLO11 is **AGPL-3.0**: if you publish code that uses it, publish it under AGPL. RTMO/rtmlib are Apache-2.0.

In [ ]:
# Config: edit these
MODEL = "yolo11m-pose.pt"   # n (fastest) / s / m / l / x
STRIDE = 1                  # process every k-th annotated frame
RUN_RTMO = False            # also extract with RTMO-m via rtmlib (Apache-2.0)
BATCH = 32
WORK = "/tmp/vicos"         # /tmp has much more space than /kaggle/working (20 GB cap)
OUT = "/kaggle/working"

In [ ]:
# Install deps (ultralytics brings torch-compatible code; Kaggle already has torch + CUDA)
!pip -q install ultralytics==8.3.* scipy
if RUN_RTMO:
    !pip -q install rtmlib onnxruntime-gpu
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Download VICOS straight into the notebook VM (10 GB zip + 209 MB json); -c resumes if interrupted
import os
os.makedirs(WORK, exist_ok=True)
!wget -q -c -O {WORK}/annotations.json https://data.vicos.si/datasets/JuiJuitsu/annotations.json
!wget -q -c -O {WORK}/images.zip https://data.vicos.si/datasets/JuiJuitsu/images.zip
!ls -lh {WORK}
!df -h /tmp | tail -1

In [ ]:
# Unzip, then delete the zip to free disk
!cd {WORK} && unzip -q -o images.zip -d images && rm images.zip
!find {WORK}/images -type f | head -5
!find {WORK}/images -type f | wc -l

In [ ]:
# Map annotation image id ("0000001") -> image file path; zip layout is discovered, not assumed
import json, glob
from pathlib import Path
with open(f"{WORK}/annotations.json") as f:
    ann = json.load(f)
files = {}
for p in glob.glob(f"{WORK}/images/**/*", recursive=True):
    if p.lower().endswith((".jpg", ".jpeg", ".png")):
        files[Path(p).stem] = p
missing = [r["image"] for r in ann if r["image"] not in files]
print(len(ann), "annotations,", len(files), "images,", len(missing), "annotations without an image file")
todo = [i for i in range(0, len(ann), STRIDE) if ann[i]["image"] in files]
print("frames to process:", len(todo))

## Matching detections to athlete identities
VICOS labels like `mount1` vs `mount2` depend on *which* athlete is which (`pose1`/`pose2`). YOLO returns people in arbitrary order,
so each detected person is matched to the original pose1/pose2 by mean keypoint distance (Hungarian algorithm, i.e. the optimal one-to-one assignment).
This uses the original annotation **only** to keep identities consistent. At inference on UFC footage, a tracker plays this role instead.

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def to_arr(p):
    return np.asarray(p, dtype=np.float32) if p and len(p) == 17 else None

def match(dets, refs, max_px=150.0):
    """dets: list of [17,3]; refs: [pose1, pose2] (or None). Returns [det_or_None, det_or_None]."""
    out = [None, None]
    ref_idx = [i for i, r in enumerate(refs) if r is not None]
    if not dets or not ref_idx:
        return out, []
    cost = np.zeros((len(ref_idx), len(dets)))
    for a, i in enumerate(ref_idx):
        r = refs[i]
        for b, d in enumerate(dets):
            ok = (r[:, 2] > 0.2) & (d[:, 2] > 0.2)
            cost[a, b] = np.linalg.norm(r[ok, :2] - d[ok, :2], axis=1).mean() if ok.sum() >= 3 else 1e6
    rows, cols = linear_sum_assignment(cost)
    dists = []
    for a, b in zip(rows, cols):
        if cost[a, b] < max_px:
            out[ref_idx[a]] = dets[b]; dists.append(cost[a, b])
    return out, dists

In [ ]:
# Run YOLO11-pose in batches and write a VICOS-format JSON
from ultralytics import YOLO
from tqdm.auto import tqdm
model = YOLO(MODEL)
new, all_d, n_det = [], [], []
for s in tqdm(range(0, len(todo), BATCH)):
    idx = todo[s:s + BATCH]
    res = model.predict([files[ann[i]["image"]] for i in idx], imgsz=640, conf=0.25,
                        verbose=False, half=True, max_det=6)
    for i, r in zip(idx, res):
        dets = []
        if r.keypoints is not None and len(r.keypoints):
            k = r.keypoints.data.cpu().numpy()          # [n, 17, 3] (x, y, conf)
            dets = [k[j] for j in range(len(k))]
        n_det.append(len(dets))
        rec = ann[i]
        m, d = match(dets, [to_arr(rec.get("pose1")), to_arr(rec.get("pose2"))])
        all_d += d
        new.append({"image": rec["image"], "frame": rec["frame"], "position": rec["position"],
                    "pose1": m[0].tolist() if m[0] is not None else None,
                    "pose2": m[1].tolist() if m[1] is not None else None})
tag = Path(MODEL).stem
out_path = f"{OUT}/annotations_{tag}_stride{STRIDE}.json"
with open(out_path, "w") as f:
    json.dump(new, f)
print("wrote", out_path)

In [ ]:
# How well does YOLO agree with the original keypoints? (a rough proxy for detection quality under grappling occlusion)
n_det = np.array(n_det)
both = sum(r["pose1"] is not None and r["pose2"] is not None for r in new)
print(f"frames: {len(new)}  |  >=2 people detected: {(n_det >= 2).mean():.1%}  |  both athletes matched: {both / len(new):.1%}")
print(f"median matched keypoint distance: {np.median(all_d):.1f} px  (90th pct {np.percentile(all_d, 90):.1f} px)")
import collections
miss = collections.Counter(r["position"] for r in new if r["pose1"] is None or r["pose2"] is None)
print("frames with a missing athlete, by position:", miss.most_common())

## Optional: RTMO-m (Apache-2.0) for comparison
Set `RUN_RTMO = True` in the config cell and re-run from the top. RTMO is a one-stage model built for crowded scenes, so it may do better on tangled bodies.

In [ ]:
if RUN_RTMO:
    import cv2
    from rtmlib import RTMO
    rtmo = RTMO(onnx_model="https://download.openmmlab.com/mmpose/v1/projects/rtmo/onnx_sdk/rtmo-m_16xb16-600e_body7-640x640-39e78cc4_20231211.zip",
                model_input_size=(640, 640), backend="onnxruntime", device="cuda")
    new_r = []
    for i in tqdm(todo):
        rec = ann[i]
        kps, scores = rtmo(cv2.imread(files[rec["image"]]))
        dets = [np.concatenate([kps[j], scores[j][:, None]], axis=1) for j in range(len(kps))]
        m, _ = match(dets, [to_arr(rec.get("pose1")), to_arr(rec.get("pose2"))])
        new_r.append({"image": rec["image"], "frame": rec["frame"], "position": rec["position"],
                      "pose1": m[0].tolist() if m[0] is not None else None,
                      "pose2": m[1].tolist() if m[1] is not None else None})
    with open(f"{OUT}/annotations_rtmo-m_stride{STRIDE}.json", "w") as f:
        json.dump(new_r, f)
    print("wrote RTMO json")

## Next
1. In the right sidebar, open **Output** → download `annotations_yolo11m-pose_stride1.json`.
2. On your laptop, save it to `data/raw/vicos_yolo/` and run:
   `.venv\Scripts\python -I src/train_baseline.py --annotations data/raw/vicos_yolo/annotations_yolo11m-pose_stride1.json --tag _yolo11m`
3. Compare `reports/metrics_lgbm_yolo11m.json` with `reports/metrics_lgbm.json`. A drop in score is the cost of switching detectors.